# CryptoPulseV2 research archive → Google Drive (one run)

**Runtime ▸ Run all.** Approve the two Google prompts (Drive access, then account access). Keep this tab open until the
last cell prints the PASS/FAIL table (about 30–45 min).

What it does:
1. Rebuilds the archive package from pinned git commits and the GDELT exports, each checked byte for byte.
2. Verifies it: checksums, frozen hashes, Parquet round trips, an offline research re-run, and comparison with the
   session's reference build.
3. Copies it into **My Drive / CryptoPulseV2_Research_Archive** in batches, `SHA256SUMS` last. It never
   overwrites or deletes; any conflict stops it.
4. Verifies the Drive copy against Google Drive's own server-side MD5 and size for every file.

It never touches Cloudflare, D1 or any CryptoPulse credential. Re-running is safe: finished work is skipped.

In [ ]:
FOLDER = "CryptoPulseV2_Research_Archive"          # the Drive folder you created (must already exist, in My Drive)
BRANCH = "claude/epic-planck-uyapsw-archive"
REPO_URL = "https://github.com/quiquandon-oss/PulseWorkerV2.git"
WORK = "/content/work"

import os, sys, json, subprocess, datetime
def run(*cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:]); raise RuntimeError(f"failed: {' '.join(cmd)}")
    return r
RESULT = {"started_utc": datetime.datetime.now(datetime.timezone.utc).isoformat()}

from google.colab import drive
drive.mount("/content/drive")
DEST = f"/content/drive/MyDrive/{FOLDER}"
assert os.path.isdir(DEST), f"Drive folder '{FOLDER}' not found in My Drive - create it there first"
print("Drive folder found; entries already inside:", len(os.listdir(DEST)))

In [ ]:
# code and tools (public repository; read-only)
if not os.path.isdir(f"{WORK}/repo/.git"):
    os.makedirs(WORK, exist_ok=True)
    run("git", "clone", "-q", "--branch", BRANCH, REPO_URL, f"{WORK}/repo")
run("git", "fetch", "-q", "origin", cwd=f"{WORK}/repo")
run("git", "checkout", "-q", BRANCH, cwd=f"{WORK}/repo"); run("git", "reset", "-q", "--hard", f"origin/{BRANCH}", cwd=f"{WORK}/repo")
RESULT["code_commit"] = run("git", "rev-parse", "HEAD", cwd=f"{WORK}/repo").stdout.strip()
pin = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow==26.0.0", "duckdb==1.5.6"], capture_output=True, text=True)
if pin.returncode != 0:
    print("pinned versions unavailable, installing current ones (Parquet bytes may then differ; rows are still checked exactly)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "duckdb"], check=True)
sys.path[:0] = [f"{WORK}/repo/research/archive", f"{WORK}/repo/research"]
import package, drive_copy
print("code", RESULT["code_commit"][:7])

In [ ]:
# 1. GDELT exports from GDELT's public server, each checked against size, MD5 and SHA-256
for attempt in range(3):
    fetched = package.fetch_gdelt(f"{WORK}/gdelt")
    print({k: (v if not isinstance(v, list) else len(v)) for k, v in fetched.items()})
    if fetched["complete"] or fetched["conflicts"]:
        break
assert fetched["complete"], "GDELT download incomplete - run this cell again"

# 2. assemble from the pinned commits, then verify
PKG = f"{WORK}/pkg"
if not os.path.isfile(f"{PKG}/{package.archive.ARCHIVE_ROOT}/SHA256SUMS"):
    package.assemble(f"{WORK}/repo", f"{WORK}/gdelt", PKG)
PKG_ROOT = f"{PKG}/{package.archive.ARCHIVE_ROOT}"
v = package.verify(PKG); c = package.compare(PKG, f"{WORK}/repo/research/archive/package_expected_sha256.json")
s = package.summary(PKG)
RESULT["package"] = {"files": sum(b["files"] for b in s["batches"].values()), "bytes": sum(b["bytes"] for b in s["batches"].values()),
                     "datasets": len(s["datasets"]), "verify_ok": v["ok"], "verify_problems": v["problems"][:10],
                     "compare_ok": c["ok"], "identical_to_reference": c["identical"], "parquet_different": len(c["parquet_different"]),
                     "different": c["different"][:10], "bundle_heads_match": c["bundle_heads_match"]}
print(json.dumps(RESULT["package"], indent=1))
assert v["ok"] and c["ok"], "package integrity FAILED - nothing was uploaded"

In [ ]:
# 3. offline research re-run from the package (network sockets disabled inside this step only)
import socket
saved = (socket.socket.connect, socket.create_connection, socket.getaddrinfo)
try:
    oc = package.offline_check(PKG, f"{WORK}/repo")
finally:
    socket.socket.connect, socket.create_connection, socket.getaddrinfo = saved
RESULT["offline_check"] = {"ok": oc["ok"], "event_replay": oc["event_replay"], "gdelt_study": oc["gdelt_study"]}
print(json.dumps(RESULT["offline_check"], indent=1))
assert oc["ok"], "offline reproducibility FAILED - nothing was uploaded"

In [ ]:
# 4. copy into Drive: batches, SHA256SUMS last, never overwrite, stop on any conflict
cp = drive_copy.copy_package(PKG_ROOT, DEST)
RESULT["upload"] = {"copied": cp["copied"], "skipped_identical": cp["skipped_identical"], "conflicts": cp["conflicts"], "batches": cp["batches"]}
print(json.dumps(RESULT["upload"], indent=1))
assert not cp["conflicts"], "CONFLICT in the Drive folder - nothing was overwritten; report the listed file(s)"
drive.flush_and_unmount()          # push every pending write to Google Drive
print("all writes flushed to Google Drive")

In [ ]:
# 5. remote verification against Google Drive's own server-side MD5 and size of every file
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
svc = build("drive", "v3")
q = f"name = '{FOLDER}' and mimeType = 'application/vnd.google-apps.folder' and 'root' in parents and trashed = false"
roots = svc.files().list(q=q, fields="files(id,name)").execute()["files"]
assert len(roots) == 1, f"expected exactly one '{FOLDER}' folder in My Drive, found {len(roots)}"
listing, stack = {}, [(roots[0]["id"], "")]
while stack:
    fid, prefix = stack.pop()
    token = None
    while True:
        page = svc.files().list(q=f"'{fid}' in parents and trashed = false", pageSize=1000, pageToken=token,
                                fields="nextPageToken, files(id,name,mimeType,md5Checksum,size)").execute()
        for f in page["files"]:
            path = prefix + f["name"]
            if f["mimeType"] == "application/vnd.google-apps.folder":
                stack.append((f["id"], path + "/"))
            else:
                listing.setdefault(path, []).append((f.get("md5Checksum"), f.get("size", -1)))
        token = page.get("nextPageToken")
        if not token:
            break
rv = drive_copy.compare_remote_listing(PKG_ROOT, listing)
RESULT["remote_verification"] = {k: (v if not isinstance(v, list) else v[:20]) for k, v in rv.items()}
print(json.dumps(RESULT["remote_verification"], indent=1))

In [ ]:
# 6. report: kept in the Drive folder under _verification/ (outside the package checksums) and shown here
drive.mount("/content/drive", force_remount=True)
RESULT["finished_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
rep_dir = f"{DEST}/{drive_copy.REPORT_DIR}"; os.makedirs(rep_dir, exist_ok=True)
name = "upload_verification_" + RESULT["finished_utc"][:19].replace(":", "") + "Z.json"
with open(f"{rep_dir}/{name}", "x") as f:
    json.dump(RESULT, f, indent=1, default=str)
drive.flush_and_unmount()
P = lambda ok: "PASS" if ok else "FAIL"
pk, up, rvf = RESULT["package"], RESULT["upload"], RESULT["remote_verification"]
print(f"Package integrity   : {P(pk['verify_ok'] and pk['compare_ok'] and RESULT['offline_check']['ok'])}  ({pk['files']} files, {pk['bytes']:,} bytes, {pk['datasets']} datasets)")
print(f"Upload              : {P(not up['conflicts'])}  (copied {up['copied']}, already present {up['skipped_identical']}, conflicts {len(up['conflicts'])})")
print(f"Remote verification : {P(rvf['ok'])}  ({rvf['files_matching']}/{rvf['files_expected']} files match Drive's MD5+size; "
      f"missing {len(rvf['missing'])}, mismatched {len(rvf['mismatched'])}, duplicates {len(rvf['duplicates'])}, unexpected {len(rvf['unexpected'])})")
print("Report saved in Drive:", f"{FOLDER}/{drive_copy.REPORT_DIR}/{name}")